In [ ]:
!pip install numpy pandas scikit-learn tensorflow

In [4]:
import pandas as pd
import os

print("=" * 70)
print("📊 PREDICTION ACCURACY TRACKER")
print("=" * 70)

# Initialize or load tracker
tracker_file = "prediction_tracker.csv"

if os.path.exists(tracker_file):
    tracker = pd.read_csv(tracker_file)
    print(f"\n✅ Loaded existing tracker with {len(tracker)} records")
else:
    tracker = pd.DataFrame(columns=[
        'Date', 'Predicted_Price', 'Actual_Price',
        'Error', 'Error_Percent', 'Direction_Correct'
    ])
    print("\n✅ Created new tracker file")


# Function to add / update prediction
def add_prediction(date, predicted_price, actual_price=None):
    global tracker  # ✅ declared ONCE and at the top

    if actual_price is None:
        new_row = {
            'Date': date,
            'Predicted_Price': predicted_price,
            'Actual_Price': None,
            'Error': None,
            'Error_Percent': None,
            'Direction_Correct': None
        }
        tracker = pd.concat([tracker, pd.DataFrame([new_row])], ignore_index=True)
        print(f"\n📝 Recording prediction for {date}: Rs. {predicted_price:.2f}")

    else:
        if date not in tracker['Date'].values:
            print(f"\n⚠️  No prediction found for {date}")
            return

        idx = tracker[tracker['Date'] == date].index[0]
        prev_price = tracker.loc[idx, 'Predicted_Price']

        error = actual_price - prev_price
        error_percent = (error / actual_price) * 100

        # Direction accuracy
        direction_correct = None
        if idx > 0 and pd.notna(tracker.loc[idx - 1, 'Actual_Price']):
            predicted_direction = prev_price > tracker.loc[idx - 1, 'Predicted_Price']
            actual_direction = actual_price > tracker.loc[idx - 1, 'Actual_Price']
            direction_correct = predicted_direction == actual_direction

        tracker.loc[idx, 'Actual_Price'] = actual_price
        tracker.loc[idx, 'Error'] = error
        tracker.loc[idx, 'Error_Percent'] = error_percent
        tracker.loc[idx, 'Direction_Correct'] = direction_correct

        print(f"\n✅ Updated {date} with actual price: Rs. {actual_price:.2f}")
        print(f"   Predicted: Rs. {prev_price:.2f}")
        print(f"   Error: Rs. {error:+.2f} ({error_percent:+.2f}%)")
        if direction_correct is not None:
            print(f"   Direction: {'✅ Correct' if direction_correct else '❌ Wrong'}")

    tracker.to_csv(tracker_file, index=False)
    print(f"💾 Saved to {tracker_file}")


# View statistics
def view_stats():
    completed = tracker[tracker['Actual_Price'].notna()]

    if completed.empty:
        print("\n⚠️  No completed predictions yet!")
        return

    print("\n" + "=" * 70)
    print("📈 PREDICTION ACCURACY STATISTICS")
    print("=" * 70)

    print(f"Total Predictions: {len(completed)}")
    print(f"Average Error: Rs. {completed['Error'].abs().mean():.2f}")
    print(f"Average Error %: {completed['Error_Percent'].abs().mean():.2f}%")
    print(f"Max Error: Rs. {completed['Error'].abs().max():.2f}")

    direction = completed['Direction_Correct'].dropna()
    if not direction.empty:
        print(f"Direction Accuracy: {direction.mean() * 100:.2f}%")

    print("\nLast 5 Predictions:")
    print(completed.tail(5).to_string(index=False))


# Main menu
def main_menu():
    while True:
        print("\n" + "=" * 70)
        print("PREDICTION TRACKER MENU")
        print("=" * 70)
        print("1. Add new prediction")
        print("2. Update with actual price")
        print("3. View statistics")
        print("4. View all records")
        print("5. Exit")

        choice = input("\nEnter choice (1-5): ").strip()

        if choice == '1':
            date = input("Enter date (YYYY-MM-DD): ")
            price = float(input("Enter predicted price: "))
            add_prediction(date, price)

        elif choice == '2':
            date = input("Enter date (YYYY-MM-DD): ")
            actual = float(input("Enter actual price: "))
            add_prediction(date, None, actual)

        elif choice == '3':
            view_stats()

        elif choice == '4':
            print("\nALL RECORDS")
            print(tracker.to_string(index=False))

        elif choice == '5':
            print("\n👋 Goodbye!")
            break

        else:
            print("\n❌ Invalid choice")


# Run
main_menu()


📊 PREDICTION ACCURACY TRACKER

✅ Created new tracker file


SyntaxError: name 'tracker' is used prior to global declaration (3863349794.py, line 73)